In [99]:
import re
import numpy as np
import pandas as pd
import os

## Base Elondou

Configuración inicial y datos

In [ ]:
path = r"C:\EDU\repositorios\AI-y-mercados-laborales-Ecuador" # Cambiar para replicar

onet = pd.read_csv(os.path.join(path,data/full_labelset.tsv), sep='\t', index_col=0)

Funciones para limpiar códigos

In [ ]:
def norm_soc(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    m = re.search(r'(\d{2})[-\.]?(\d{4})', x)
    if m:
        return f{m.group(1)}-{m.group(2)}
    return np.nan

def norm_ciuo(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace(',', '.')
    m = re.search(r'(\d{4})', x)
    if m:
        return m.group(1)
    return np.nan

Creamos scores basados en humanos de forma explícita

In [102]:
def human_to_scores(label):
    if label == 'E1':
        return pd.Series([1.0, 1.0, 1.0])
    elif label == 'E2':
        return pd.Series([0.0, 0.5, 1.0])
    else:
        return pd.Series([0.0, 0.0, 0.0])

onet[['alpha_h', 'beta_h', 'gamma_h']] = onet['human_exposure_agg'].apply(human_to_scores)

Agregamos rúbricas deepseek

In [104]:
deep = pd.read_csv(os.path.join(path, r"code\deepseek api\deepseek_agg.csv"))

In [106]:
deep = deep.rename(columns={
        'task_id':'Task ID',
        'pE0_mean':'deep_pE0_mean',
        'pE1_mean':'deep_pE1_mean',
        'pE2_mean':'deep_pE2_mean',
        'pE0_sd':'deep_pE0_sd',
        'pE1_sd':'deep_pE1_sd',
        'pE2_sd':'deep_pE2_sd',
        'alpha_soft_mean':'deep_alpha_soft_mean',
        'beta_soft_mean':'deep_beta_soft_mean',
        'zeta_soft_mean':'deep_zeta_soft_mean',
        'hard_label_mean_probs':'deepseek_exposure',
        'mean_confidence':'deep_mean_confidence',
        'n_runs':'deep_n_runs',
    })

In [107]:
onet = pd.merge(onet, deep, on='Task ID')

In [109]:
onet.columns

Index(['O*NET-SOC Code', 'Task ID', 'Task', 'Task Type', 'Title',
       'human_exposure_agg', 'gpt4_exposure', 'gpt4_exposure_alt_rubric',
       'gpt_3_relevant', 'gpt4_automation', 'alpha', 'beta', 'gamma',
       'automation', 'human_labels', 'alpha_h', 'beta_h', 'gamma_h',
       'onet_soc', 'title', 'task_type', 'deep_pE0_mean', 'deep_pE1_mean',
       'deep_pE2_mean', 'deep_pE0_sd', 'deep_pE1_sd', 'deep_pE2_sd',
       'deep_alpha_soft_mean', 'deep_beta_soft_mean', 'deep_zeta_soft_mean',
       'deepseek_exposure', 'deep_mean_confidence', 'deep_n_runs'],
      dtype='object')

Agregamos SOC

In [110]:
onet['soc_code'] = onet['O*NET-SOC Code'].apply(norm_soc)
onet['ocupaciones ONET'] = onet['Title']
onet['task_weight'] = np.where(onet['Task Type'].eq('Core'), 2.0, 1.0)

soc_exp = (
    onet.groupby(['soc_code', 'ocupaciones ONET'], as_index=False)
        .apply(lambda g: pd.Series({
            'alpha_gpt': np.average(g['alpha'], weights=g['task_weight']),
            'beta_gpt': np.average(g['beta'], weights=g['task_weight']),
            'gamma_gpt': np.average(g['gamma'], weights=g['task_weight']),
            'alpha_h': np.average(g['alpha_h'], weights=g['task_weight']),
            'beta_h': np.average(g['beta_h'], weights=g['task_weight']),
            'gamma_h': np.average(g['gamma_h'], weights=g['task_weight']),
            'alpha_deep':np.average(g['deep_alpha_soft_mean'], weights=g['task_weight']),
            'beta_deep':np.average(g['deep_beta_soft_mean'], weights=g['task_weight']),
            'zeta_deep':np.average(g['deep_zeta_soft_mean'], weights=g['task_weight']),
            'n_tasks': g['Task ID'].nunique()
        }))
        .reset_index(drop=True)
)

C:\Users\OSCARJ\AppData\Local\Temp\ipykernel_76200\3641388844.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


## Crosswalk

In [57]:
cw_own = pd.read_csv(os.path.join(path, "crosswalk/crosswalkOwn.csv"), sep=',', encoding='latin1', dtype=str)
cw_bls = pd.read_csv(os.path.join(path, "crosswalk/crosswalkBLS.csv"), sep="\t", dtype=str)

In [111]:
cw_bls['soc_code'] = cw_bls['2010 SOC Code'].apply(norm_soc)
cw_bls['ciuo_code'] = cw_bls['ISCO-08 Code'].apply(norm_ciuo)

cw_own['Código CIUO'] = cw_own['Código CIUO'].apply(norm_ciuo)

Esposición CIUO

In [112]:
soc_ciuo = cw_bls.merge(soc_exp, on='soc_code', how='left')
soc_ciuo2 = cw_own.merge(soc_exp, on='ocupaciones ONET', how='left')

In [113]:
ciuo_exp = (
    soc_ciuo.groupby('ciuo_code', as_index=False)
    .agg({
        'alpha_gpt': 'mean',
        'beta_gpt': 'mean',
        'gamma_gpt': 'mean',
        'alpha_h': 'mean',
        'beta_h': 'mean',
        'gamma_h': 'mean',
        'soc_code': 'nunique',
        'alpha_deep':'mean',
        'beta_deep':'mean',
        'zeta_deep':'mean'
    })
    .rename(columns={'soc_code': 'n_soc_matches'})
)

ciuo_exp2 = (
    soc_ciuo2.groupby('Código CIUO', as_index=False)
    .agg({
        'alpha_gpt': 'mean',
        'beta_gpt': 'mean',
        'gamma_gpt': 'mean',
        'alpha_h': 'mean',
        'beta_h': 'mean',
        'gamma_h': 'mean',
        'soc_code': 'nunique',
        'alpha_deep':'mean',
        'beta_deep':'mean',
        'zeta_deep':'mean'
    })
    .rename(columns={'soc_code': 'n_soc_matches'})
)

## Merge ENEMDU

In [114]:
path_enemdu = r"Z:\survey\ECU\ENEMDU\2025\m12\data_orig\enemdu_persona_2025_12.dta" # Cambiar para replicar

enemdu = pd.read_stata(path_enemdu, convert_categoricals=False)

In [115]:
enemdu['ciuo_code'] = enemdu['p40'].apply(norm_ciuo)
enemdu['Código CIUO'] = enemdu['p40'].apply(norm_ciuo)
enemdu_exp = enemdu.merge(ciuo_exp, on='ciuo_code', how='left')
enemdu_exp2 = enemdu.merge(ciuo_exp2, on='Código CIUO', how='left')

C:\Users\OSCARJ\AppData\Local\Temp\ipykernel_76200\1312861203.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enemdu['ciuo_code'] = enemdu['p40'].apply(norm_ciuo)
C:\Users\OSCARJ\AppData\Local\Temp\ipykernel_76200\1312861203.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enemdu['Código CIUO'] = enemdu['p40'].apply(norm_ciuo)


In [118]:
enemdu_exp.to_stata(os.path.join(path, r'data\enemdu_con_tareas_lbs.parquet'))
enemdu_exp.to_stata(os.path.join(path, r'data\enemdu_con_tareas_own.parquet'))

C:\Users\OSCARJ\AppData\Local\Temp\ipykernel_76200\238781939.py:1: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    Código CIUO   ->   C_digo_CIUO

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  enemdu_exp.to_stata(os.path.join(path, r'data\enemdu_con_tareas_lbs.parquet'))
C:\Users\OSCARJ\AppData\Local\Temp\ipykernel_76200\238781939.py:2: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    Código CIUO   ->   C_digo_CIUO

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  enemdu_exp.to_stata(os.path.join(path, r'data\enemdu_con_ta